# Model_Pruning
L1 unstructured pruning on the best MobileNetV2 checkpoint, then fine-tune.

Run **Model_MobileNetV2** first.

In [ ]:
# ── Mount Drive & load utils ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/stm32-thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place the utils/ folder at: My Drive/stm32-thesis/utils/")

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────
import os, time, random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune

from utils.dataset import prepare_dataset, get_loaders
from utils.models  import VWW_MobileNetV2
from utils.train   import setup_device, set_seed, evaluate, train_epoch, validate_epoch, plot_history

device = setup_device(seed=41)

In [ ]:
prepare_dataset()
train_loader, val_loader = get_loaders(batch_size=64, augmentation="standard")

In [ ]:
# ── Config ──────────────────────────────────────────────────────────
SAVE_DIR     = "/content/drive/My Drive/stm32-thesis/checkpoints"
CKPT         = f"{SAVE_DIR}/mobilenetv2_seed_85.pth"   # adjust seed to your best
SAVE_PATH    = f"{SAVE_DIR}/mobilenetv2_pruned_30pct.pth"

PRUNE_AMOUNT = 0.30
FT_EPOCHS    = 10
FT_LR        = 1e-4

In [ ]:
# ── Load model and baseline accuracy ────────────────────────────────
model = VWW_MobileNetV2().to(device)
model.load_state_dict(torch.load(CKPT, map_location=device))

base_acc = evaluate(model, val_loader, device)
print(f"Baseline val acc: {base_acc*100:.2f}%")

In [ ]:
# ── Apply L1 unstructured pruning ────────────────────────────────────
layers = [(m, "weight") for m in model.modules() if isinstance(m, nn.Conv2d)]
for layer, param in layers:
    prune.l1_unstructured(layer, name=param, amount=PRUNE_AMOUNT)

post_prune_acc = evaluate(model, val_loader, device)
print(f"After pruning val acc (before fine-tune): {post_prune_acc*100:.2f}%")

In [ ]:
# ── Fine-tune with val tracking and best-checkpoint saving ───────────
set_seed(41)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=FT_LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FT_EPOCHS)

best_ft_acc   = 0.0
best_ft_state = None
start         = time.time()

for epoch in range(1, FT_EPOCHS + 1):
    tl, ta = train_epoch(model, train_loader, optimizer, criterion, device)
    _,  va = validate_epoch(model, val_loader, criterion, device)
    scheduler.step()

    marker = " ✅" if va > best_ft_acc else ""
    print(f"FT {epoch:2d}/{FT_EPOCHS} | Train {ta*100:.2f}% | Val {va*100:.2f}%{marker}")

    if va > best_ft_acc:
        best_ft_acc   = va
        best_ft_state = {k: v.clone() for k, v in model.state_dict().items()}

# Restore best, make pruning permanent, save
model.load_state_dict(best_ft_state)
for layer, param in layers:
    prune.remove(layer, param)
torch.save(model.state_dict(), SAVE_PATH)

print(f"\nFine-tune done in {(time.time()-start)/60:.1f} min")
print(f"Saved: {SAVE_PATH}")

In [ ]:
# ── Summary ──────────────────────────────────────────────────────────
final_acc = evaluate(model, val_loader, device)

print("="*55)
print(f"Baseline val acc          : {base_acc*100:.2f}%")
print(f"After pruning (before FT) : {post_prune_acc*100:.2f}%")
print(f"After fine-tune val acc   : {best_ft_acc*100:.2f}%")
print(f"Accuracy delta            : {(best_ft_acc - base_acc)*100:+.2f}%")
print("="*55)

# Sparsity check
total = zeroed = 0
for m in model.modules():
    if isinstance(m, nn.Conv2d):
        w = m.weight.detach().cpu().numpy()
        total += w.size; zeroed += (w == 0).sum()
print(f"Model sparsity: {zeroed/total*100:.1f}%")